# VNWoodKnot Qualitative Figure (Colab, Hybrid I/O)

This notebook rebuilds the publication-ready VNWoodKnot qualitative figure for the target-domain analysis.

It compares:
- Ground truth
- `T0`: target-only training
- `T1`: source-initialized fine-tuning from `Y0-3600e200`

Hybrid strategy:
- keep the full processed dataset on Google Drive
- create a symlink at `/content/processed_for_server`
- export the small VNWoodKnot YOLO dataset locally to `/content/local_data`
- train `T0` and `T1`
- dump raw predictions
- build the final qualitative figure from raw images + GT + predictions

This notebook assumes the `Y0-e200` checkpoint already exists in Drive, typically produced by the in-domain notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi


In [ ]:
from pathlib import Path
import os

REPO_URL = 'https://github.com/ntkhanh98/wood-defect-q2.git'
REPO_DIR = Path('/content/wood-defect-q2')

# Update if your Drive path differs.
SOURCE_PATH = Path('/content/drive/MyDrive/2.Work/1.PTIT/1.Cá nhân/2.Research/2026/wood-defected-q2/data/processed_for_server')
DEST_PATH = Path('/content/processed_for_server')

VN_ROOT = DEST_PATH / 'vnwoodknot'
LOCAL_VN_YOLO_ROOT = Path('/content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo')
ARTIFACT_ROOT = Path('/content/drive/MyDrive/wood_q2_vn_artifacts')
Y0_CHECKPOINT = Path('/content/drive/MyDrive/wood_q2_in_domain_artifacts/yolo/y0_yolov8s_vsb7_3600_rarefirst_e200/weights/best.pt')

IMAGE_SIZE = 1024
EPOCHS = 50

os.environ['WOOD_VN_PROCESSED_ROOT'] = str(VN_ROOT)
os.environ['PYTHONUNBUFFERED'] = '1'

print('SOURCE_PATH   =', SOURCE_PATH)
print('DEST_PATH     =', DEST_PATH)
print('VN_ROOT       =', VN_ROOT)
print('LOCAL_VN_YOLO =', LOCAL_VN_YOLO_ROOT)
print('ARTIFACT_ROOT =', ARTIFACT_ROOT)
print('Y0_CHECKPOINT =', Y0_CHECKPOINT)


In [ ]:
%%bash
set -euo pipefail

cd /content
rm -rf wood-defect-q2
git clone "$REPO_URL" wood-defect-q2
cd wood-defect-q2

python3 -m pip install -q ultralytics==8.3.0 pandas pillow pyyaml


In [ ]:
%%bash
set -euo pipefail

rm -rf /content/processed_for_server || true
ln -sfn "$SOURCE_PATH" "$DEST_PATH"
echo "Linked $SOURCE_PATH -> $DEST_PATH"


In [ ]:
for p in [
    SOURCE_PATH,
    VN_ROOT / 'manifest.jsonl',
    VN_ROOT / 'metadata.json',
    REPO_DIR / 'scripts' / 'build_vnwoodknot_qualitative_from_predictions.py',
    REPO_DIR / 'scripts' / 'evaluate_yolov8.py',
    Y0_CHECKPOINT,
]:
    print(p, 'OK' if p.exists() else 'MISSING')

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
(ARTIFACT_ROOT / 'yolo').mkdir(exist_ok=True)
(ARTIFACT_ROOT / 'tables').mkdir(exist_ok=True)
(ARTIFACT_ROOT / 'figures').mkdir(exist_ok=True)
LOCAL_VN_YOLO_ROOT.parent.mkdir(parents=True, exist_ok=True)


## Export VNWoodKnot YOLO dataset locally

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

rm -rf /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo || true

python3 scripts/build_yolo_dataset.py \
  --input-manifest /content/processed_for_server/vnwoodknot/manifest.jsonl \
  --image-root-dir /content/processed_for_server/vnwoodknot \
  --output-root-dir /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo \
  --dataset-name vnwoodknot_live_dead_2class_yolo \
  --classes live_knot dead_knot \
  --copy-images


In [ ]:
for p in [
    VN_ROOT / 'manifest.jsonl',
    Path('/content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/dataset.yaml'),
]:
    print(p, 'OK' if p.exists() else 'MISSING')


## Train `T0` and `T1`

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/train_yolov8.py \
  --data /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/dataset.yaml \
  --model yolov8s \
  --experiment-name t0_yolov8s_vnwoodknot_target_only_e50 \
  --epochs 50 \
  --imgsz 1024 \
  --batch 32 \
  --device 0 \
  --workers 4 \
  --seed 42 \
  --patience 30 \
  --project-dir /content/drive/MyDrive/wood_q2_vn_artifacts/yolo

python3 scripts/train_yolov8.py \
  --data /content/local_data/vnwoodknot/benchmarks/vnwoodknot_live_dead_2class_yolo/dataset.yaml \
  --weights /content/drive/MyDrive/wood_q2_in_domain_artifacts/yolo/y0_yolov8s_vsb7_3600_rarefirst_e200/weights/best.pt \
  --experiment-name t1_y0_3600e200_to_vnwoodknot_e50 \
  --epochs 50 \
  --imgsz 1024 \
  --batch 32 \
  --device 0 \
  --workers 4 \
  --seed 42 \
  --patience 30 \
  --project-dir /content/drive/MyDrive/wood_q2_vn_artifacts/yolo


## Evaluate on the held-out test split and save raw predictions

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/dataset_transfer.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_vn_artifacts/yolo/t0_yolov8s_vnwoodknot_target_only_e50/weights/best.pt \
  --experiment-name t0_yolov8s_vnwoodknot_target_only_e50_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_vn_artifacts \
  --save-predictions

python3 scripts/evaluate_yolov8.py \
  --dataset-config configs/dataset_transfer.yaml \
  --checkpoint /content/drive/MyDrive/wood_q2_vn_artifacts/yolo/t1_y0_3600e200_to_vnwoodknot_e50/weights/best.pt \
  --experiment-name t1_y0_3600e200_to_vnwoodknot_e50_eval \
  --split test \
  --batch 8 \
  --imgsz 1024 \
  --device 0 \
  --output-dir /content/drive/MyDrive/wood_q2_vn_artifacts \
  --save-predictions


## Build the final VNWoodKnot qualitative figure

In [ ]:
%%bash
set -euo pipefail

cd /content/wood-defect-q2

python3 scripts/build_vnwoodknot_qualitative_from_predictions.py \
  --manifest /content/processed_for_server/vnwoodknot/manifest.jsonl \
  --split test \
  --rows 4 \
  --t0-run-name t0_yolov8s_vnwoodknot_target_only_e50 \
  --t0-header T0 \
  --t0-predictions /content/drive/MyDrive/wood_q2_vn_artifacts/tables/t0_yolov8s_vnwoodknot_target_only_e50_eval_test_predictions.jsonl \
  --t1-run-name t1_y0_3600e200_to_vnwoodknot_e50 \
  --t1-header T1 \
  --t1-predictions /content/drive/MyDrive/wood_q2_vn_artifacts/tables/t1_y0_3600e200_to_vnwoodknot_e50_eval_test_predictions.jsonl \
  --output-dir /content/drive/MyDrive/wood_q2_vn_artifacts/figures/vnwoodknot_transfer_qualitative_raw


In [ ]:
%%bash
set -euo pipefail

find /content/drive/MyDrive/wood_q2_vn_artifacts/figures/vnwoodknot_transfer_qualitative_raw -maxdepth 1 -type f | sort
